# 08 — Score and analyze the four memory conditions

This notebook uses the benchmark's intended-reading and figure-meaning fields to create saved preliminary labels with a Vertex model judge. These are model-judge labels, not human adjudication; audit the saved reasons before presenting completion/error claims as final.

Because there is one complete run per condition, this notebook reports means but not standard deviations.

The original judge does not receive the actual supplied data or retrieved memory. Its labels need manual audit. Episode membership is only a relevance proxy. Task success with retrieval does not establish causal knowledge transfer. See the README for the full pilot limitations.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'src' / 'state_management').is_dir() else WORKING_DIR.parent
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
RUNS_DIR = EXPERIMENTS_DIR / 'runs'
MANIFEST_PATH = RUNS_DIR / 'tasks.jsonl'
LABELS_PATH = RUNS_DIR / 'evaluation_labels.jsonl'
RUNNER_PATH = EXPERIMENTS_DIR / 'score_vertex_results.py'
print('This scoring pass makes up to 48 paid Vertex judge requests, one per saved answer.')

In [ ]:
environment = os.environ.copy()
environment.setdefault('VERTEX_AI_PROJECT_ID', 'dsi-tifin-harness')
environment.setdefault('VERTEX_AI_LOCATION', 'global')
environment.setdefault('VERTEX_MODEL', 'gemini-3.5-flash-lite')
run = subprocess.run([sys.executable, str(RUNNER_PATH), str(MANIFEST_PATH), str(RUNS_DIR), str(LABELS_PATH)], cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True, check=False)
print(run.stdout)
if run.returncode:
    print(run.stderr)
    raise RuntimeError('Scoring stopped. Existing labels were preserved; re-run to resume.')

In [ ]:
from statistics import mean

RUN_FILES = {'none': 'vertex_results.jsonl', 'raw_trace': 'raw_trace_results.jsonl', 'summary': 'summary_results.jsonl', 'structured_lesson': 'structured_lesson_results.jsonl'}
task_metadata = {task['task_id']: task for task in (json.loads(line) for line in MANIFEST_PATH.read_text(encoding='utf-8').splitlines() if line)}
records = {condition: [json.loads(line) for line in (RUNS_DIR / filename).read_text(encoding='utf-8').splitlines() if line] for condition, filename in RUN_FILES.items()}
for items in records.values():
    for item in items:
        item.update({key: task_metadata[item['task_id']][key] for key in ('episode', 'turn')})
labels = {label['run_id']: label for label in (json.loads(line) for line in LABELS_PATH.read_text(encoding='utf-8').splitlines() if line)}
assert all(len(items) == 12 for items in records.values())
assert len(labels) == 48

def rate(numerator, denominator):
    return {'numerator': numerator, 'denominator': denominator, 'pct': round(100 * numerator / denominator, 1) if denominator else None}

summary = {}
for condition, items in records.items():
    retrievals = [item for item in items if item.get('retrieved_memory_ids')]
    related = [item for item in items if item['turn'] in (2, 3)]
    stale = [item for item in items if item['episode'] == 'stale_data']
    completion = rate(sum(labels[item['run_id']]['completed'] for item in items), 12)
    repeated = rate(sum(labels[item['run_id']]['repeated_error'] for item in related), len(related))
    precision = rate(sum(item.get('retrieval_relevant') is True for item in retrievals), sum(len(item.get('retrieved_memory_ids', [])) for item in retrievals))
    transfer = None if condition == 'none' else rate(sum(labels[item['run_id']]['completed'] and item.get('retrieval_relevant') is True for item in related), len(related))
    stale_rate = rate(sum(labels[item['run_id']]['stale_memory_error'] for item in stale), len(stale))
    summary[condition] = {
        'completion': completion, 'tool_calls_mean': mean(item['tool_calls'] for item in items),
        'input_tokens_mean': mean(item['input_tokens'] for item in items), 'output_tokens_mean': mean(item['output_tokens'] for item in items),
        'total_tokens_mean': mean(item['total_tokens'] for item in items), 'latency_seconds_mean': mean(item['latency_ms'] for item in items) / 1000,
        'repeated_error': repeated, 'retrieval_precision': precision if retrievals else None, 'knowledge_transfer': transfer,
        'stale_memory_error': stale_rate, 'retrieval_overhead_ms_mean': mean(item['retrieval_overhead_ms'] for item in retrievals) if retrievals else None,
        'standard_deviation': 'not reported: one complete run per condition',
    }
summary

In [ ]:
baseline = summary['none']
comparisons = {}
for condition, metrics in summary.items():
    if condition == 'none':
        continue
    comparisons[condition] = {}
    for metric in ('tool_calls_mean', 'input_tokens_mean', 'output_tokens_mean', 'total_tokens_mean', 'latency_seconds_mean', 'retrieval_overhead_ms_mean'):
        value, base = metrics[metric], baseline[metric]
        comparisons[condition][metric] = None if value is None or base in (None, 0) else {'absolute_change': value - base, 'relative_change_pct': 100 * (value - base) / base}
    for metric in ('completion', 'repeated_error', 'stale_memory_error'):
        value, base = metrics[metric], baseline[metric]
        comparisons[condition][metric] = {'absolute_change_pct_points': value['pct'] - base['pct'], 'relative_change_pct': None if base['pct'] == 0 else 100 * (value['pct'] - base['pct']) / base['pct']}
(RUNS_DIR / 'analysis_metrics.json').write_text(json.dumps({'metrics': summary, 'vs_none': comparisons}, indent=2), encoding='utf-8')
print(json.dumps({'metrics': summary, 'vs_none': comparisons}, indent=2))